In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# %run /Workspace/Users/gayatrijoshi663@gmail.com/regis-healthcare/1_setup/utility

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","discharges","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
df_silver = df_silver.withColumn(
    "discharge_id",
    F.trim(F.col("discharge_id"))
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "discharge_date",
    F.trim(F.col("discharge_date"))
).withColumn(
    "discharge_reason",
    F.trim(F.col("discharge_reason"))
).withColumn(
    "destination",
    F.trim(F.col("destination"))
).withColumn(
    "authorized_by",
    F.trim(F.col("authorized_by"))
).withColumn(
    "notes",
    F.trim(F.col("notes"))
).withColumn(
    "notes",
    F.trim(F.col("notes"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# discharge_id

from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("discharge_id").rlike("^DIS"))

df_silver = df_silver.withColumn(
    "discharge_id",
    when(
        (col("discharge_id").isNull()) | (~col("discharge_id").rlike("^DIS")),
        "0"
    ).otherwise(col("discharge_id"))
)

display(df_filt)
display(df_silver)

In [0]:
# resident_id
from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("resident_id").rlike("^RES"))

df_silver = df_silver.withColumn(
    "resident_id",
    when(
        (col("resident_id").isNull()) | (~col("resident_id").rlike("^RES")),
        "0"
    ).otherwise(col("resident_id"))
)

display(df_filt)
display(df_silver)

In [0]:
# facility_id 
from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("facility_id").rlike("^FAC"))

df_silver = df_silver.withColumn(
    "facility_id",
    when(
        (col("facility_id").isNull()) | (~col("facility_id").rlike("^FAC")),
        "0"
    ).otherwise(col("facility_id"))
)

display(df_filt)
display(df_silver)

In [0]:
# discharge_date

from pyspark.sql.functions import to_timestamp
df_silver = df_silver.withColumn("discharge_date",when(~col("discharge_date").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"),None).otherwise(col("discharge_date")))

df_invalid = df_silver.filter(~col("discharge_date").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)
df_invalid = df_silver.groupBy("discharge_date").count().filter(col("count")>1)
display(df_invalid)

filt_xx = {'9999-99-99':None,
           'not-a-date' : None,
           '2030-01-01': None,
           '2026-02-30':None}
df_silver = df_silver.replace(filt_xx,subset = ["discharge_date"])

df_silver = df_silver.withColumn("discharge_date",to_timestamp(col("discharge_date"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)

In [0]:
# discharge_reason

from pyspark.sql.functions import col, when, initcap, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn(
    "discharge_reason",
    when(col("discharge_reason").isNull() |
        initcap(trim(col("discharge_reason"))).isin(invalid_values),lit("not provided")
    ).otherwise(initcap(trim(col("discharge_reason"))))
)
# display(df_silver)
dup= df_silver.groupBy("discharge_reason").count()
display(dup)

In [0]:
# destination
from pyspark.sql.functions import col, when, initcap, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn(
    "destination",
    when(col("destination").isNull() |
        initcap(trim(col("destination"))).isin(invalid_values),lit("UNKNOWN")
    ).otherwise(initcap(trim(col("destination"))))
)
# display(df_silver)
dup= df_silver.groupBy("destination").count()
display(dup)

In [0]:
# authorized_by
from pyspark.sql.functions import col, when, lower, trim, lit

df_filt = df_silver.filter(~col("authorized_by").rlike("^Dr"))
display(df_filt)

invalid_values ={ "null" : "Not Defined", 
                  "nan" : "Not Defined",
                   "n/a" : "Not Defined",
                  "#n/a" : "Not Defined",
                   "none" : "Not Defined",
                    "" : "Not Defined"}
df_silver = df_silver.replace(invalid_values,subset = ["authorized_by"])
display(df_silver)

In [0]:
# notes
from pyspark.sql.functions import col, when, initcap, trim, lit
dd_sf = df_silver.filter(~col("notes").rlike("^Discharge"))
# display(dd_sf)
dup = dd_sf.groupBy("notes").count()
# display(dup)
invalid_values = ["null", "nan", "n/a", "#n/a", "none",
"NULL",
"#N/A",
"UNKNOWN",
"NaN",
"null",
"NONE",
"N/A",
"",
"Null",
"Not Given",
"Unknown",
"Nan",
"None",
"N/a"]

df_silver = df_silver.withColumn(
    "notes",
    when(col("notes").isNull() |
        initcap(trim(col("notes"))).isin(invalid_values),lit("Not Given")
    ).otherwise(initcap(trim(col("notes"))))
)
# display(df_silver)
dup= df_silver.groupBy("notes").count()
display(dup)

In [0]:
# created_at
dup= df_silver.groupBy("created_at").count()
# display(dup)

df_silver = df_silver.withColumn("created_at",to_timestamp(col("created_at"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)

#### Silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")